In [ ]:
pip install unidecode

In [ ]:
import unidecode
import pandas as pd
import sys
sys.path.append('..')
import re
import unicodedata

from extract.scrape_html_rule.skills_dict import SKILLS

In [ ]:
data = pd.read_json('../data/scrape_html_rule.json')

In [ ]:
data.head()

In [ ]:
df_skills_job = data.explode("skills")
df_skills_job.head()

In [ ]:
SOFT_SKILLS = {
    "Comunicacao",
    "Trabalho em Equipe",
    "Proatividade",
    "Resolucao de Problemas",
    "Pensamento Analitico",
    "Organizacao",
    "Autonomia",
    "Lideranca",
    "Curiosidade",
    "Storytelling",
    "Visao Estrategica",
}
def rank_skill(skill: str) -> str:
    return "soft_skill" if skill in SOFT_SKILLS else "technical_skill"

In [ ]:
df_skills_job["type_skill"] = df_skills_job["skills"].apply(
    lambda x: "soft_skill" if x in SOFT_SKILLS else "technical_skill"
)
df_skills_job.head()

In [ ]:
df_skills_job['title'].unique()

In [ ]:
CATEGORIAS = {
    "Data Analyst", "Data Engineer", "BI Analyst", "Data Intern",
    "Analytics Engineer", "Data Scientist", "Data Architect",
    "Data Leadership", "Data Product", "Data Governance",
    "Data Talent Pool", "Not Data Role",
}


def normalize_position(position):
    # se ja vier normalizado, devolve sem reprocessar
    if str(position) in CATEGORIAS:
        return str(position)

    p = remover_acentos(str(position).lower().strip())
    # remove pontuacao que quebra o "in" (parenteses, barras, pipes, etc.)
    p = re.sub(r"[()/|.,]", " ", p)
    p = re.sub(r"\s+", " ", p).strip()

    # =========================
    # BANCO DE TALENTOS  (antes de tudo: e generico, nao um cargo)
    # =========================
    if "banco de talentos" in p:
        return "Data Talent Pool"

    # =========================
    # NAO E CARREIRA DE DADOS  (antes das regras de dados p/ nao roubar match)
    # =========================
    if any(x in p for x in [
        "analista de ia",
        "consultor ti - ia",
        "consultor ti ia",
        "analista de monitoramento",
        "prevencao de perdas",
        "analista de sustentabilidade",
        "analista administrativo",
        "relacoes institucionais",
        "analista de melhoria continua",
        "analista de gestao",
        "analista sistema de gestao",
        "analista de controle e gestao",
        "analista de cadastro",
        "analista nto",
        "engenheiro de sistemas",
        "analista de sistemas jr",
        "coletor de dados",
        "bolsista de inovacao",
        "assistente de qualidade",
        "analista de atrativos",
        "analista jr de projetos de engenharia",
        "servicos externos",
    ]):
        return "Not Data Role"

    # =========================
    # DATA LEADERSHIP
    # =========================
    if any(x in p for x in [
        "gerente de dados",
        "lead dados",
        "lead de dados",
        "data lead",
        "head of data",
        "tech lead",
        "data engineering lead",
        "data analytics coordinator",
        "data science coordinator",
        "coordenador de dados",
        "coordenador de dados de mercado",
        "engineering manager",
        "data leadership",
    ]):
        return "Data Leadership"

    # =========================
    # DATA PRODUCT
    # =========================
    if any(x in p for x in [
        "data product owner",
        "data product manager",
        "data products specialist",
    ]):
        return "Data Product"

    # =========================
    # DATA ARCHITECT
    # =========================
    if any(x in p for x in [
        "data architect",
        "cloud data architect",
        "arquiteto de dados",
    ]):
        return "Data Architect"

    # =========================
    # DATA GOVERNANCE
    # =========================
    if any(x in p for x in [
        "governanca de dados",
        "data governance",
        "commercial data governance analyst",
    ]):
        return "Data Governance"

    # =========================
    # DATA SCIENTIST
    # =========================
    if any(x in p for x in [
        "data scientist",
        "staff data scientist",
        "data science",
        "cientista de dados",
        "cientista dados",
        "ciencia de dados",
        "machine learning",
        "mlops",
        "artificial intelligence",
        "inteligencia artificial",
        "genai",
        "generative ai",
        "ai engineer",
        "especialista em ciencia de dados",
        "forward deployed data scientist",
        "python engineer - machine learning focus",
        "analista data science dados",
    ]):
        return "Data Scientist"

    # =========================
    # ANALYTICS ENGINEER
    # =========================
    if any(x in p for x in [
        "analytics engineer",
        "analytics engineer iii",
        "analytics engineer junior",
        "staff analytics engineer",
        "senior analytics engineer",
        "data analytics engineer",
        "data analyst engineer",
        "engenharia de analytics",
        "engenheiro de analytics",
        "engenharia de dados & analytics",
        "engenharia de dados e analytics",
        "plataforma de analytics",
    ]):
        return "Analytics Engineer"

    # =========================
    # DATA ENGINEER
    # =========================
    if any(x in p for x in [
        "data engineer",
        "senior data engineer",
        "junior data engineer",
        "staff data engineer",
        "data engineer specialist",
        "data engineer consultant",
        "data engineer trainee",
        "engenheiro de dados",
        "engenheira de dados",         # pessoa engenheira / engenheiro(a)
        "engenheiro a de dados",       # vem de "engenheiro(a)" apos limpeza
        "engenharia de dados",
        "eng de dados",
        "engenheiro de banco de dados",
        "engenheiro de dados azure",
        "engenheiro de dados azure databricks",
        "engenheiro solucoes dados",
        "data solutions",
        "engenheiro de confiabilidade de banco de dados",
        "etl engineer",
        "etl data engineer",
        "desenvolvimento de etl",
        "dados e etl",
        "snowflake engineer",
        "data platform engineer",
        "senior data platform engineer",
        "plataforma de dados",
        "dataops engineer",
        "database reliability engineer",
        "senior database reliability engineer",
        "big data engineer",
        "data quality engineer",
        "data solutions engineer",
        "data management engineer",
        "data engineering specialist",
        "especialista engenharia de dados",
        "especialista engenharia dados",
        "especialista em engenharia de dados",
        "suporte de banco de dados",
        "suporte de banco dados",
        "dev banco dados",
        "dev banco de dados",
        "databricks",
    ]):
        return "Data Engineer"

    # =========================
    # BI / ANALYTICS
    # =========================
    if any(x in p for x in [
        "business intelligence",
        "business intelligence analyst",
        "business intelligence specialist",
        " bi ",
        "de bi",
        "a bi",
        "power bi",
        "desenvolvedor de bi",
        "inteligencia comercial",
        "inteligencia de mercado",
        "inteligencia de negocios",
        "inteligencia competitiva",
        "inteligencia operacional",
        "people analytics",
        "data analytics",
        "data & analytics",
        "data visualization",
        "data visualization specialist",
        "consultor dados business intelligence",
        "martech analyst",
        "analytics intern",
        "specialista em analytics",
        "indicadores",
    ]):
        return "BI Analyst"

    # =========================
    # DATA ANALYST
    # =========================
    if any(x in p for x in [
        "analista de dados",
        "analista dados",
        "analistas de dados",          # plural
        "data analyst",
        "data & ai analyst",
        "finance data analyst",
        "fraud data analyst",
        "industrial data analyst",
        "business data analyst",
        "hr data analyst",
        "etl data analyst",
        "datahub analyst",
        "data migration analyst",
        "data automation analyst",
        "data development analyst",
        "data intelligence analyst",
        "market intelligence analyst",
        "senior data analyst",
        "dados junior",
        "dados pleno",
        "dados senior",
        "pleno de dados",
        "senior de dados",
        "junior de dados",
        "planejamento dados",
        "planejamento e dados",
        "operacoes de dados",
        "projetos de dados",
        "projetos e dados",
        "validacao de dados",
        "migracao de dados",
        "inteligencia de dados",
        "especialista dados",
        "especialista em dados",
        "especialista de dados",
        "finance data specialist",
        "analista de modelagem de dados",
        "analista de base de dados",
        "analista de governanca de dados",
        "analista de dados dba",
        "coordenador de dados de mercado",
        "ti e dados",
        "logistico de dados",
        "analise de dados",
        "foco em dados",
        "data & ai",
    ]):
        return "Data Analyst"

    # =========================
    # INTERNS
    # =========================
    if any(x in p for x in [
        "intern",
        "estagi",
        "stagiaire",
        "trainee",
    ]):
        return "Data Intern"

    # =========================
    # NAO E CARREIRA DE DADOS (fallback conservador)
    # =========================
    return "Not Data Role"

In [ ]:
df_skills_job["position_normalized"] = (
    df_skills_job["title"]
    .str.lower()
    .apply(unidecode.unidecode)
    .str.strip()
)
df_skills_job.head()

In [ ]:
df_skills_job["position_group"] = (
    df_skills_job["position_normalized"]
    .fillna("")
    .apply(normalize_position)
)

In [ ]:
df_skills_job['position_group'].unique()

In [ ]:
df_skills_job['position_group']

In [ ]:
import unidecode

skills_df["position_normalized"] = (
    skills_df["position"]
    .str.lower()
    .apply(unidecode.unidecode)
    .str.strip()
)

In [ ]:
skills_df["position_normalized"].unique()

In [ ]:
def normalize_position(position):
    p = unidecode(str(position).lower().strip())

    # =========================
    # DATA LEADERSHIP
    # =========================
    if any(x in p for x in [
        "gerente de dados",
        "lead dados",
        "data lead",
        "head of data",
        "tech lead",
        "data engineering lead",
        "data analytics coordinator",
        "data science coordinator",
        "coordenador de dados",
        "coordenador de dados de mercado",
        "engineering manager"
    ]):
        return "Data Leadership"

    # =========================
    # DATA PRODUCT
    # =========================
    if any(x in p for x in [
        "data product owner",
        "data product manager",
        "data products specialist"
    ]):
        return "Data Product"

    # =========================
    # DATA ARCHITECT
    # =========================
    if any(x in p for x in [
        "data architect",
        "cloud data architect"
    ]):
        return "Data Architect"

    # =========================
    # DATA GOVERNANCE
    # =========================
    if any(x in p for x in [
        "governanca de dados",
        "data governance",
        "commercial data governance analyst"
    ]):
        return "Data Governance"

    # =========================
    # DATA SCIENTIST
    # =========================
    if any(x in p for x in [
        "data scientist",
        "staff data scientist",
        "data science",
        "machine learning",
        "mlops",
        "artificial intelligence",
        "inteligencia artificial",
        "genai",
        "generative ai",
        "ai engineer",
        "especialista em ciencia de dados",
        "forward deployed data scientist",
        "python engineer - machine learning focus",
        "analista data science dados"
    ]):
        return "Data Scientist"

    # =========================
    # ANALYTICS ENGINEER
    # =========================
    if any(x in p for x in [
        "analytics engineer",
        "analytics engineer iii",
        "analytics engineer junior",
        "staff analytics engineer",
        "senior analytics engineer",
        "data analytics engineer",
        "data analyst engineer"
    ]):
        return "Analytics Engineer"

    # =========================
    # DATA ENGINEER
    # =========================
    if any(x in p for x in [
        "data engineer",
        "senior data engineer",
        "junior data engineer",
        "staff data engineer",
        "data engineer specialist",
        "data engineer consultant",
        "data engineer trainee",
        "engenheiro de dados",
        "engenheiro de banco de dados",
        "engenheiro de dados azure",
        "engenheiro de dados azure databricks",
        "engenheiro solucoes dados",
        "engenheiro de confiabilidade de banco de dados",
        "etl engineer",
        "etl data engineer",
        "snowflake engineer",
        "data platform engineer",
        "senior data platform engineer",
        "dataops engineer",
        "database reliability engineer",
        "senior database reliability engineer",
        "big data engineer",
        "data quality engineer",
        "data solutions engineer",
        "data management engineer",
        "data engineering specialist",
        "especialista engenharia de dados",
        "especialista em engenharia de dados",
        "suporte de banco de dados",
        "dev banco dados",
        "databricks"
    ]):
        return "Data Engineer"

    # =========================
    # BI / ANALYTICS
    # =========================
    if any(x in p for x in [
        "business intelligence",
        "business intelligence analyst",
        "business intelligence specialist",
        "inteligencia comercial",
        "inteligencia de mercado",
        "inteligencia de negocios",
        "inteligencia competitiva",
        "people analytics",
        "data analytics",
        "data visualization",
        "data visualization specialist",
        "consultor dados business intelligence",
        "martech analyst",
        "analytics intern",
        "specialista em analytics",
        "indicadores"
    ]):
        return "BI Analyst"

    # =========================
    # DATA ANALYST
    # =========================
    if any(x in p for x in [
        "analista de dados",
        "analista dados",
        "data analyst",
        "finance data analyst",
        "fraud data analyst",
        "industrial data analyst",
        "business data analyst",
        "hr data analyst",
        "etl data analyst",
        "datahub analyst",
        "data migration analyst",
        "data automation analyst",
        "data development analyst",
        "data intelligence analyst",
        "market intelligence analyst",
        "senior data analyst",
        "dados junior",
        "dados pleno",
        "dados senior",
        "planejamento dados",
        "operacoes de dados",
        "projetos de dados",
        "validacao de dados",
        "inteligencia de dados",
        "especialista dados",
        "especialista em dados",
        "finance data specialist",
        "analista de modelagem de dados",
        "analista de base de dados",
        "analista de governanca de dados",
        "analista de dados dba",
        "coordenador de dados de mercado"
    ]):
        return "Data Analyst"

    # =========================
    # INTERNS
    # =========================
    if any(x in p for x in [
        "intern",
        "estagi",
        "stagiaire",
        "trainee"
    ]):
        return "Data Intern"

    # =========================
    # NÃO É CARREIRA DE DADOS
    # =========================
    return None

In [ ]:
skills_df["position_group"] = (
    skills_df["position_normalized"]
    .fillna("")
    .apply(normalize_position)
)

In [ ]:
positions = (
    skills_df
    .groupby("position_group")["job_id"]
    .nunique()
    .reset_index(name="total")
    .sort_values("total", ascending=False)
)
positions

In [ ]:
skills_df["level"].unique()

In [ ]:
levels = (
  skills_df.
  groupby("level")["job_id"].
  nunique().
  reset_index(name="total").
  sort_values("total", ascending=False)
)
levels

In [ ]:
mapping = {
  "junior": "entry",
  "assistente": "assistant",
  "junior": "entry",
  "pleno": "mid"
}

skills_df["level_grouped"] = (
  skills_df["level"].replace(mapping)
)
levels = (
  skills_df.
  groupby("level_grouped")["job_id"].
  nunique().
  reset_index(name="total").
  sort_values("total", ascending=False)
)
levels

In [ ]:
skills_df.head()

In [ ]:
skills = (
    skills_df
    .groupby("skill")["job_id"]
    .nunique()
    .reset_index(name="total")
    .sort_values("total", ascending=False)
)
skills

In [ ]:
import unidecode
skills_df["skill_normalized"] = (
    skills_df["skill"]
    .str.lower()
    .apply(unidecode.unidecode)
    .str.strip()
)

In [ ]:
from unidecode import unidecode

def standardize_skill(skill):
    # Converte para string, remove acentos, joga para minúsculo e tira espaços extras
    s = unidecode(str(skill).lower().strip())

    # =========================================================
    # 1. MAPEAMENTO EXATO (Dicionário de Sinônimos/Erros comuns)
    # =========================================================
    # Tudo aqui deve estar em minúsculo e sem acento
    exact_matches = {
        "cosmosdb": "Azure Cosmos DB",
        "azure cosmos db": "Azure Cosmos DB",
        "cosmos db": "Azure Cosmos DB",
        
        "powerbi": "Power BI",
        "power bi": "Power BI",
        "microsoft power bi": "Power BI",
        
        "excel": "Microsoft Excel",
        "microsoft excel": "Microsoft Excel",
        "excel avancado": "Microsoft Excel",
        "excel intermediario": "Microsoft Excel",
        
        "aws": "AWS",
        "amazon web services": "AWS",
        "amazon web services (aws)": "AWS",
        
        "gcp": "Google Cloud Platform",
        "google cloud": "Google Cloud Platform",
        "google cloud platform (gcp)": "Google Cloud Platform",
        
        "postgres": "PostgreSQL",
        "postgresql": "PostgreSQL",
        
        "ms sql server": "SQL Server",
        "sql server": "SQL Server",
        "mssql": "SQL Server",
        "microsoft sql server": "SQL Server",
        
        "phyton": "Python", # Erro de digitação comum
        "python": "Python",
        
        "pyspark": "PySpark",
        "spark": "Apache Spark",
        "apache spark": "Apache Spark",
        
        "dbt (data build tool)": "dbt",
        "dbt core": "dbt",
        "dbt": "dbt",
        
        "llms": "LLM",
        "llm": "LLM",
        "large language models (llms)": "LLM",
        "large language models": "LLM"
    }

    # Se a string normalizada for uma chave exata, retorna o valor oficial
    if s in exact_matches:
        return exact_matches[s]

    # =========================================================
    # 2. MAPEAMENTO POR SUBSTRING (Pega variações maiores)
    # =========================================================
    # A ordem importa aqui. Coloque termos mais específicos primeiro.
    
    if "cosmos" in s:
        return "Azure Cosmos DB"
        
    if "databricks" in s:
        return "Databricks" # Vai capturar "Databricks Jobs", "Azure Databricks", etc.
        
    if "bigquery" in s or "big query" in s:
        return "Google BigQuery"
        
    if "machine learning" in s:
        return "Machine Learning"
        
    if "comunicacao" in s:
        return "Comunicação" # Unifica "boa comunicacao", "comunicacao verbal", etc.

    # =========================================================
    # 3. RETORNO PADRÃO
    # =========================================================
    # Se não cair em nenhuma regra de unificação, retorna a string original 
    # apenas formatada em Title Case (Primeiras Letras Maiúsculas) para ficar limpo.
    return str(skill).strip().title()

In [ ]:
skills_df["skill_grouped"] = (
    skills_df["skill_normalized"]
    .fillna("")
    .apply(standardize_skill)
)
skills = (
    skills_df
    .groupby("skill_grouped")["job_id"]
    .nunique()
    .reset_index(name="total")
    .sort_values("total", ascending=False)
)
skills

In [ ]:
import json

skills = pd.DataFrame(
    skills_df["skill"].unique(),
    columns=["skill"]
)
with open("skills.json", "w", encoding="utf-8") as f:
    json.dump(
        skills.to_dict(orient="records"),
        f,
        ensure_ascii=False,
        indent=2
    )

In [ ]:
skills_df.to_csv("../data/skills_standardized.csv", index=False)

In [ ]:
skills = (
    skills_df
    .groupby("skill_grouped")["job_id"]
    .nunique()
    .reset_index(name="total")
    .sort_values("total", ascending=False)
)
with open("skills.json", "w", encoding="utf-8") as f:
    json.dump(
        skills.to_dict(orient="records"),
        f,
        ensure_ascii=False,
        indent=2
    )